# load all

### packages and functions

In [ ]:
import os
import pickle
import numpy as np
import mne
import itertools

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from scipy.stats import sem, wilcoxon, ttest_1samp
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2_contingency

In [ ]:
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [ ]:
import random

In [ ]:
# statistical annotation
def convert_pvalue_to_asterisks(pvalue):
    if pvalue <= 0.0001:
        return " **** "
    elif pvalue <= 0.001:
        return " *** "
    elif pvalue <= 0.01:
        return "  **  "
    elif pvalue <= 0.05:
        return "   *  "
    return "  ns  "

In [ ]:
cm = 1/2.54 # figure size from inch to cm

In [ ]:
pe_list_1 = ['nonPE', 'SPE', 'RPE']
pe_list_2 = ['nonPE', 'SPE', 'RPE', 'dualPE']

cmap_pe = {'SPE':'#ff9eaa', 'RPE':'#3aa6b9', 'nonPE': '#808080', 'dualPE':'#6aa84f',\
           'same': '#deac80', 'diff':'#982b1c'}

In [ ]:
def cal_mean_sem(values):
    values = [x for x in values if ~np.isnan(x)]
    # Calculate the mean
    mean_value = np.mean(values)
    # Calculate the SEM
    sem_value = sem(values)

    return mean_value, sem_value

In [ ]:
def l1_norm(series):
    return series/series.sum()

In [ ]:
def apply_mask(array, mask):
    return [
        [array[i][j] if mask[i][j] == 1 else None for j in range(len(array[i]))]
        for i in range(len(array))
    ]

### data and configs

In [ ]:
subj_list = ["03", "04", "05", "07", "08", "10","12", "13", "14", "15", "16", "17"] # 

In [ ]:
raw_dir = '../../data/eeg'
cp_dir = '../../data/decoder_gradCAM'

In [ ]:
with open(raw_dir+'/A03_tfr_1.pkl', 'rb') as f:
    tfr = pickle.load(f)
    tfr.apply_baseline((None,None),mode='ratio', verbose='error')
ch_names = tfr.ch_names
info = tfr.info

In [ ]:
band_list = ['δ\n[1-4Hz]','θ\n[4-8Hz]','α\n[8-14Hz]','β\n[14-31Hz]','γ\n[31-49Hz]']
band_list_simple = ['δ','θ','α','β','γ']

# Compute gradCAMs

## Binary choice task

In [ ]:
cp_dir_1 = f'{cp_dir}'
raw_dir_1 = f'{raw_dir}/raw_epochs_binary'

### loading

In [ ]:
with open(f'{cp_dir_1}/gradCAM_binary.pkl', "rb") as f:  # Python 3: open(..., 'wb')
        all_test_data_1, all_true_label_1, all_test_cam_1, \
              all_pred_label_1, all_prob_label_1 = pickle.load(f)

### create df

In [ ]:
# group cam with different correct/wrong pred_label - cam
# every channel has similar cam so check ['E8'] only TODO: why?
all_correct_cam = []
for s in range(len(subj_list)):
    f_correct_cam = []
    for f in range(5):
        data = all_test_data_1[s][f]
        e_correct_cam_0 = []
        e_correct_cam_1 = []
        e_correct_cam_2 = []
        for e in range(len(data)):
            if all_pred_label_1[s][f][e] == all_true_label_1[s][f][e]:
                # normalize cam
                data = all_test_cam_1[s][f][e]
                this_cam = data/data.sum()

                if all_true_label_1[s][f][e] == 0:
                    e_correct_cam_0.append(this_cam)
                elif all_true_label_1[s][f][e] == 1:
                    e_correct_cam_1.append(this_cam)
                elif all_true_label_1[s][f][e] == 2:
                    e_correct_cam_2.append(this_cam)
        f_correct_cam.append([e_correct_cam_0, e_correct_cam_1, e_correct_cam_2])
    all_correct_cam.append(f_correct_cam)

In [ ]:
# calculate all cams for each participant
all_cam = []
for s in range(len(subj_list)):
    for f in range(5):
        for b in range(5):
            if not np.isnan(np.mean(all_correct_cam[s][f][0],axis=0)).any():
                for e in range(len(all_correct_cam[s][f][0])):
                    for ch in range(len(all_correct_cam[s][f][0][e])):
                        all_cam.append({'subj':subj_list[s], 'fold':f, \
                                        'cat': 'nonPE', 'ch':ch,\
                                            'bank': b, 'e_id':e, \
                                            'cam': all_correct_cam[s][f][0][e][ch][b]})
            if not np.isnan(np.mean(all_correct_cam[s][f][1],axis=0)).any():
                for e in range(len(all_correct_cam[s][f][1])):
                     for ch in range(len(all_correct_cam[s][f][1][e])):
                        all_cam.append({'subj':subj_list[s], 'fold':f, \
                                        'cat': 'SPE', 'ch':ch,\
                                            'bank': b, 'e_id':e, \
                                             'cam': all_correct_cam[s][f][1][e][ch][b]})
            if not np.isnan(np.mean(all_correct_cam[s][f][2],axis=0)).any():
                for e in range(len(all_correct_cam[s][f][2])):
                     for ch in range(len(all_correct_cam[s][f][2][e])):
                        all_cam.append({'subj':subj_list[s], 'fold':f, \
                                        'cat': 'RPE', 'ch':ch,\
                                            'bank': b, 'e_id':e, \
                                            'cam': all_correct_cam[s][f][2][e][ch][b]})

In [ ]:
all_cam_df = pd.DataFrame(all_cam)

In [ ]:
# mean gradCAMs per subject, norm_cam: normalized across frequency bands
subj_mean_cam = all_cam_df.groupby(['subj','cat','bank','ch'])['cam'].mean()
subj_mean_cam_df = subj_mean_cam.reset_index()
subj_mean_cam_df['norm_cam'] = subj_mean_cam_df.groupby(['subj','cat','ch'])['cam'].transform(l1_norm)

In [ ]:
# grand averaged gradCAM
mean_cam = all_cam_df.groupby(['cat','bank','ch'])['cam'].mean()
mean_cam_df = mean_cam.reset_index()

In [ ]:
# grand averaged gradCAM, norm_cam: normalized across bands
norm_mean_cam = all_cam_df.groupby(['cat','bank','ch'])['cam'].mean()
norm_mean_cam_df = norm_mean_cam.reset_index()
norm_mean_cam_df['norm_cam'] = norm_mean_cam_df.groupby(['cat','ch'])['cam'].transform(l1_norm)

### simple visualization

In [ ]:
fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(3, 5))
pe_list = pe_list_1
for p in range(len(pe_list)):
    df = mean_cam_df[mean_cam_df['cat']==pe_list[p]]
    for f in range(5):
        this_cam = df[df['bank']==f]['cam']
        im, _ = mne.viz.plot_topomap(np.array(this_cam), \
                                    tfr.pick_channels(ch_names).info,\
                                    sensors=False,
                                    # mask=np.array([0]*1+[1]+[0]*3+[1]+[0]*(64-8)),
                                    # mask=sig_mask, mask_params=mask_params,\
                                    extrapolate='head',image_interp='nearest',\
                                    # sphere=(0.0, 0.0, 0.0, 0.18),\
                                    cmap='Reds',\
                                    vlim = (mean_cam_df['cam'].min(),\
                                        mean_cam_df['cam'].max()),\
                                axes=axes[f, p], show=False)
plt.tight_layout()
plt.show()

### stats analysis

In [ ]:
# for all channels
all_peak_freq_1 = []
all_peak_p_value_1 = []

for cat in pe_list_1:
    pe_peak_1 = []
    pe_p_value_1 = []
    this_df = subj_mean_cam_df[subj_mean_cam_df['cat']==cat]

    for n_ch in range(len(ch_names)):
        df = this_df[this_df['ch']==n_ch]
        # Step 1: Identify the position of the maximum value for each subject
        max_positions = df.loc[df.groupby('subj')['norm_cam'].idxmax(), 'bank'].values

        # Step 2: Count the frequency of each position (1 to 5)
        position_counts = np.bincount(max_positions, minlength=5)

        # Step 3: Expected frequency if distribution is random
        expected_counts = np.array([len(df['subj'].unique()) / 5] * 5)

        # Step 4: Perform Chi-Square Goodness-of-Fit Test
        chi2_stat, p_value = chi2_contingency([position_counts, expected_counts])[:2]
        pe_p_value_1.append(p_value)
        if p_value<0.05:
            pe_peak_1.append(np.argmax(position_counts))
        else:
            pe_peak_1.append(None)

    all_peak_p_value_1.append(pe_p_value_1)
    all_peak_freq_1.append(pe_peak_1)

In [ ]:
# correction of multiple tests
p_before = all_peak_p_value_1
p_before = np.array(p_before) # 5 region, 5 freq, 2 comparison
org_shape = p_before.shape
_, ps_corrected, _, _ = multipletests(p_before.flatten(), method='fdr_bh')
ps_corrected = ps_corrected.reshape(org_shape)
all_peak_p_value_1 = ps_corrected
all_peak_freq_1 = np.array(all_peak_freq_1)
all_peak_freq_1 = apply_mask(all_peak_freq_1,all_peak_p_value_1<0.05)

## Navigation task

In [ ]:
cp_dir_2 = f'{cp_dir}'
raw_dir_2 = f'{raw_dir}/raw_epochs_navigation'

#### loading

In [ ]:
with open(f'{cp_dir_2}/gradCAM_navigation.pkl', "rb") as f:  # Python 3: open(..., 'wb')
        all_test_data_2, all_true_label_2, all_test_cam_2, \
                all_prob_label_2, all_pred_label_2 = pickle.load(f)

### create df

In [ ]:
# group cam with different correct/wrong pred_label - cam

# extract all correctly prredicted ones
all_correct_cam_2 = []
for s in range(len(subj_list)):
    f_correct_cam = []
    for f in range(5):
        data = all_test_data_2[s][f]
        e_correct_cam_0 = []
        e_correct_cam_1 = []
        e_correct_cam_2 = []
        e_correct_cam_3 = []
        for e in range(len(data)):
            if all_pred_label_2[s][f][e] == all_true_label_2[s][f][e]:
                # normalize cam
                data = all_test_cam_2[s][f][e]
                this_cam = data/data.sum()
                if all_true_label_2[s][f][e] == 0:
                    e_correct_cam_0.append(this_cam)
                elif all_true_label_2[s][f][e] == 1:
                    e_correct_cam_1.append(this_cam)
                elif all_true_label_2[s][f][e] == 2:
                    e_correct_cam_2.append(this_cam)
                elif all_true_label_2[s][f][e] == 3:
                    e_correct_cam_3.append(this_cam)
        f_correct_cam.append([e_correct_cam_0, e_correct_cam_1, e_correct_cam_2, e_correct_cam_3])
    all_correct_cam_2.append(f_correct_cam)

In [ ]:
# calculate all cams for each participant
all_cam_2 = []
for s in range(len(subj_list)):
    for f in range(5):
        for b in range(5):
            if not np.isnan(np.mean(all_correct_cam_2[s][f][0],axis=0)).any():
                for e in range(len(all_correct_cam_2[s][f][0])):
                    for ch in range(len(all_correct_cam_2[s][f][0][e])):
                        all_cam_2.append({'subj':subj_list[s], 'fold':f, \
                                          'cat': 'nonPE', 'ch': ch,\
                                            'bank': b, 'e_id':e, \
                                        'cam': all_correct_cam_2[s][f][0][e][ch][b]})
            if not np.isnan(np.mean(all_correct_cam_2[s][f][1],axis=0)).any():
                for e in range(len(all_correct_cam_2[s][f][1])):
                    for ch in range(len(all_correct_cam_2[s][f][1][e])):
                        all_cam_2.append({'subj':subj_list[s], 'fold':f, \
                                          'cat': 'SPE', 'ch': ch,\
                                            'bank': b, 'e_id':e, \
                                         'cam': all_correct_cam_2[s][f][1][e][ch][b]})
            if not np.isnan(np.mean(all_correct_cam_2[s][f][2],axis=0)).any():
                for e in range(len(all_correct_cam_2[s][f][2])):
                    for ch in range(len(all_correct_cam_2[s][f][2][e])):
                        all_cam_2.append({'subj':subj_list[s], 'fold':f, \
                                          'cat': 'RPE', 'ch': ch,\
                                            'bank': b, 'e_id':e, \
                                        'cam': all_correct_cam_2[s][f][2][e][ch][b]})
            if not np.isnan(np.mean(all_correct_cam_2[s][f][3],axis=0)).any():
                for e in range(len(all_correct_cam_2[s][f][3])):
                    for ch in range(len(all_correct_cam_2[s][f][3][e])):
                        all_cam_2.append({'subj':subj_list[s], 'fold':f, \
                                          'cat': 'dualPE', 'ch': ch,\
                                            'bank': b, 'e_id':e, \
                                         'cam': all_correct_cam_2[s][f][3][e][ch][b]})

In [ ]:
all_cam_2_df = pd.DataFrame(all_cam_2)

In [ ]:
# mean gradCAMs per subject, norm_cam: normalized across frequency bands
subj_mean_2_cam = all_cam_2_df.groupby(['subj','cat','ch','bank'])['cam'].mean()
subj_mean_cam_2_df = subj_mean_2_cam.reset_index()
subj_mean_cam_2_df['norm_cam'] = subj_mean_cam_2_df.groupby(['subj','cat','ch'])['cam'].transform(l1_norm)

In [ ]:
# grand averaged gradCAM
mean_2_cam = all_cam_2_df.groupby(['cat','ch','bank'])['cam'].mean()
mean_cam_2_df = mean_2_cam.reset_index()
mean_cam_2_df['norm_cam'] = mean_cam_2_df.groupby(['cat'])['cam'].transform(l1_norm)

In [ ]:
# grand averaged gradCAM, norm_cam: normalized across bands
norm_mean_2_cam = all_cam_2_df.groupby(['cat','bank','ch'])['cam'].mean()
norm_mean_cam_2_df = norm_mean_2_cam.reset_index()
norm_mean_cam_2_df['norm_cam'] = norm_mean_cam_2_df.groupby(['cat','ch'])['cam'].transform(l1_norm)

In [ ]:
subj_mean_2_cam = all_cam_2_df.groupby(['subj','cat','bank','ch'])['cam'].mean()
subj_mean_cam_2_df = subj_mean_2_cam.reset_index()
subj_mean_cam_2_df['norm_cam'] = subj_mean_cam_2_df.groupby(['subj','cat','ch'])['cam'].transform(l1_norm)

### simple visualization

In [ ]:
fig, axes = plt.subplots(nrows=5, ncols=4, figsize=(8*cm, 10*cm))
pe_list = pe_list_2
for p in range(len(pe_list)):
    df = mean_cam_2_df[mean_cam_2_df['cat']==pe_list[p]]
    for f in range(5):
        this_cam = df[df['bank']==f]['cam']
        im, _ = mne.viz.plot_topomap(this_cam, \
                                    info,\
                                    sensors=False,
                                    # mask=sig_mask, mask_params=mask_params,\
                                    extrapolate='box',image_interp='nearest',\
                                    # sphere=(0.0, 0.0, 0.0, 0.10),\
                                    cmap='Reds',\
                                    vlim = (mean_cam_2_df['cam'].min(),\
                                        mean_cam_2_df['cam'].max()),
                                axes=axes[f,p], show=False)
        


plt.tight_layout(pad=0.1, w_pad=0, h_pad=.6)
plt.subplots_adjust(left=0.01, right=0.99, top=0.8, bottom=0)
plt.gcf().set_dpi(300)
plt.savefig('../../results/figS3a.png')
plt.show()

### stats analysis

In [ ]:
# for all channels
all_peak_freq_2 = []
all_peak_p_value_2 = []

for cat in pe_list_2:
    pe_peak_2 = []
    pe_p_value_2 = []
    this_df = subj_mean_cam_2_df[subj_mean_cam_2_df['cat']==cat]

    for n_ch in range(len(ch_names)):
        df = this_df[this_df['ch']==n_ch]
        # Step 1: Identify the position of the maximum value for each subject
        max_positions = df.loc[df.groupby('subj')['norm_cam'].idxmax(), 'bank'].values

        # Step 2: Count the frequency of each position (1 to 5)
        position_counts = np.bincount(max_positions, minlength=5)

        # Step 3: Expected frequency if distribution is random
        expected_counts = np.array([len(df['subj'].unique()) / 5] * 5)

        # Step 4: Perform Chi-Square Goodness-of-Fit Test
        chi2_stat, p_value = chi2_contingency([position_counts, expected_counts])[:2]

        pe_p_value_2.append(p_value)
        if p_value<0.05:
            pe_peak_2.append(np.argmax(position_counts))
        else:
            pe_peak_2.append(None)

    all_peak_p_value_2.append(pe_p_value_2)
    all_peak_freq_2.append(pe_peak_2)

In [ ]:
p_before = all_peak_p_value_2
p_before = np.array(p_before) # 5 region, 5 freq, 2 comparison
org_shape = p_before.shape
_, ps_corrected, _, _ = multipletests(p_before.flatten(), method='fdr_bh')
ps_corrected = ps_corrected.reshape(org_shape)
all_peak_p_value_2 = ps_corrected

## both tasks

In [ ]:
mean_cam_df['game'] = 'Game1'
mean_cam_2_df['game'] = 'Game2'

norm_mean_cam_df['game'] = 'Game1'
norm_mean_cam_2_df['game'] = 'Game2'

subj_mean_cam_df['game'] = 'Game1'
subj_mean_cam_2_df['game'] = 'Game2'

all_cam_df['game'] = 'Game1'
all_cam_2_df['game'] = 'Game2'

cam_df = pd.concat([all_cam_df, all_cam_2_df])

# Analysis and Visualization

## Cross-task decoding performance

In [ ]:
pe_list_tf = ['r10', 's10']
tf_list = ['bcTOna', 'naTObc']
result_dir = '../../data/decoder_results'

In [ ]:
with open(result_dir+'/tf_learning_results.pkl', 'rb') as f:
    measures_tf_df = pickle.load(f)

In [ ]:
tf_title_list = ['RPE/nonPE', 'SPE/nonPE']
palette = ['#7E60BF', '#433878']
cm = 1/2.54 

fig,ax = plt.subplots(figsize=(5*cm,4.2*cm))
df = measures_tf_df
# df = df[df['so_type']!='sub']
sns.pointplot(ax=ax, data=df, x='pe_type', y='auc',\
                hue='game', hue_order=tf_list,\
                dodge=.4,join=False,capsize=.1, palette=palette,\
                errorbar='ci', errwidth=.9, scale=.4)


# scatter for each subject
for n in range(len(pe_list_tf)):
    this_df = df[df['pe_type']==pe_list_tf[n]]
    a = -1
    for g in tf_list:
        pe_data = this_df[(this_df['game'] == g)]
        ax.scatter([n+0.2*a]*len(pe_data['auc']), pe_data['auc'], \
                   label=pe_list_tf[n],\
                    edgecolors='None', alpha=.2, color='grey', s=10)
        a = 1


    # Manually adding lines between the error bars for the same x
    pos1 = n + 0.2*(-1)
    pos2 = n + 0.2*(1)
    mean1 = this_df[(this_df['game'] == tf_list[0])]['auc'].mean()
    mean2 = this_df[(this_df['game'] == tf_list[1])]['auc'].mean()
    ax.plot([pos1, pos2], [mean1, mean2], color='black', linestyle='-', linewidth=.8)

    # Plot lines connecting the same so_pe points for each subject
    for subj in df['subj'].unique():
        subj_data = this_df[this_df['subj'] == subj]
        # for so in df['so_type'].unique():
        c1_data = subj_data[subj_data['game'] == tf_list[0]]
        c2_data = subj_data[subj_data['game'] == tf_list[1]]
        if len(c1_data) >= 1 and len(c2_data)>=1:
            ax.plot([n-0.2,n+0.2], \
                    [c1_data['auc'],c2_data['auc']], color='grey', alpha=.2,\
                        linestyle='--', linewidth=.8)


ax.set_xlim([-.5, 1.5])

ax.axhline(y=.5,color='grey', alpha=.7, linestyle='--')
ax.set_xticks(ax.get_xticks(), tf_title_list, fontsize=8)
ax.set_yticks(ax.get_yticks(), ax.get_yticklabels(),fontsize=8)
ax.set_ylim([.498, .63])

ax.set_xlabel('', fontsize=8)
ax.set_ylabel('AUROC', fontsize=8)
handles, labels = ax.get_legend_handles_labels()  # Get current legend handles and labels
ax.legend(handles[:2], tf_list, title='', title_fontsize=8, fontsize=8)


plt.suptitle('Transfer learning', y=1,fontsize=8, fontweight='bold')
plt.tight_layout(pad=0.15, w_pad=1.0,h_pad=2.0)
# plt.subplots_adjust(left=0.1, right=0.9, top=0.8, bottom=0.2)
plt.gcf().set_dpi(300)
plt.savefig('../../results/fig5a.png')
plt.show()

In [ ]:
df = measures_tf_df
auc_list = []
plist = []
for pe, g in itertools.product(pe_list_tf, tf_list):
    print(g, pe)
    this_df = df[(df['pe_type']==pe)&(df['game']==g)]
    print(cal_mean_sem(this_df['auc']))
    _, p = ttest_1samp(this_df['auc'], .5,alternative='two-sided')
    auc_list.append(this_df['auc'])
    plist.append(p)

_,p = wilcoxon(auc_list[0], auc_list[1])
plist.append(p)
_,p = wilcoxon(auc_list[2], auc_list[3])
plist.append(p)

print(plist)
multipletests(plist, method='fdr_bh')

## Similarity

#### load functions

In [ ]:
from skimage.metrics import structural_similarity as ssim
from sklearn.metrics import confusion_matrix

In [ ]:
def compute_ssim_mse(matrix1, matrix2):
    mssim = []
    mse = []

    for a in matrix1.tolist():
        for b in matrix2.tolist():
            a = np.array(a)
            b = np.array(b)
            
            mssim.append(ssim(a, b, win_size=3, data_range=np.max([a,b])-np.min([a,b])))
            mse.append(np.mean((a-b) ** 2))
        
    return np.mean(mssim), np.mean(mse)

In [ ]:
def compute_sim_map(matrix1, matrix2):
    sim_maps = []


    for a in matrix1.tolist():
        for b in matrix2.tolist():
            a = np.array(a)
            b = np.array(b)
            _,S = ssim(a, b, win_size=3, data_range=np.max([a,b])-np.min([a,b]), full=True)
            sim_maps.append(S)
            
        
    return sim_maps

#### binary choice task: sim_diff_1_df

In [ ]:
# similarity acros categories (between different categories), about 30s
df = all_cam_df
subj_list = list(df['subj'].unique())

# Group by 'subj' and 'cat'
grouped = df.groupby(['subj', 'cat'])

# Extract data for each category
category_data = {}
for (subj, cat), group in grouped:
    group['f_e_id'] = group['fold'].astype(str) + '_' + group['e_id'].astype(str)
    group['c_b'] = group['ch'].astype(str) + '_' + group['bank'].astype(str)
    pivot_table = group.pivot(index='f_e_id', columns='c_b', values='cam').fillna(0)
    data_matrix = pivot_table.values
    category_data[(subj, cat)] = data_matrix.reshape(data_matrix.shape[0],len(ch_names),len(band_list))

# Compute similarity between different categories
results = []

# Compare each category with every other category
for (subj1, cat1), data_matrix1 in category_data.items():
    for (subj2, cat2), data_matrix2 in category_data.items():
        if subj1 == subj2:
            # Compute similarities between different categories
            mssim, mse = compute_ssim_mse(data_matrix1, data_matrix2)

            
            # Get confusion matrix for accuracy
            s1 = subj_list.index(subj1)
            s2 = subj_list.index(subj2)
            this_pe_list = pe_list_1
            pe1 = this_pe_list.index(cat1)
            pe2 = this_pe_list.index(cat2)
            
            cm1 = confusion_matrix(y_true=[item for l in all_true_label_1[s1] for item in l],
                                   y_pred=[item for l in all_pred_label_1[s1] for item in l],
                                   labels=[0, 1, 2], normalize='true')
            
            cm2 = confusion_matrix(y_true=[item for l in all_true_label_1[s2] for item in l],
                                   y_pred=[item for l in all_pred_label_1[s2] for item in l],
                                   labels=[0, 1, 2], normalize='true')
            
            acc1 = cm1[pe1][pe1]
            acc2 = cm2[pe2][pe2]
            
            # Store results
            if cat1==cat2:
                cat0 = 'same'
            else:
                cat0='diff-' + cat2 
            results.append({'subj': subj1, 'cat1': cat1, 'cat2': cat2, 'cat-condition':cat0, \
                            'sim': mssim, 'mse': mse, 'acc1': acc1, 'acc2': acc2})


# Convert results to DataFrame for easier manipulation
sim_diff_1_df = pd.DataFrame(results)

#### navigation task: sim_diff_2_df

In [ ]:
# similarity acros categories (between different categories), about 1min
df = all_cam_2_df
subj_list = list(df['subj'].unique())

# Group by 'subj' and 'cat'
grouped = df.groupby(['subj', 'cat'])

# Extract data for each category
category_data = {}
for (subj, cat), group in grouped:
    group['f_e_id'] = group['fold'].astype(str) + '_' + group['e_id'].astype(str)
    group['c_b'] = group['ch'].astype(str) + '_' + group['bank'].astype(str)
    pivot_table = group.pivot(index='f_e_id', columns='c_b', values='cam').fillna(0)
    data_matrix = pivot_table.values
    category_data[(subj, cat)] = data_matrix.reshape(data_matrix.shape[0],len(ch_names),len(band_list))

# Compute similarity between different categories
results_2 = []

# Compare each category with every other category
for (subj1, cat1), data_matrix1 in category_data.items():
    for (subj2, cat2), data_matrix2 in category_data.items():
        if subj1 == subj2:
            # Compute similarities between different categories
            mssim, mse = compute_ssim_mse(data_matrix1, data_matrix2)
            
            # Get confusion matrix for accuracy
            s1 = subj_list.index(subj1)
            s2 = subj_list.index(subj2)
            this_pe_list = pe_list_2
            pe1 = this_pe_list.index(cat1)
            pe2 = this_pe_list.index(cat2)
            
            cm1 = confusion_matrix(y_true=[item for l in all_true_label_2[s1] for item in l],
                                   y_pred=[item for l in all_pred_label_2[s1] for item in l],
                                   labels=[0, 1, 2, 3], normalize='true')
            
            cm2 = confusion_matrix(y_true=[item for l in all_true_label_2[s2] for item in l],
                                   y_pred=[item for l in all_pred_label_2[s2] for item in l],
                                   labels=[0, 1, 2, 3], normalize='true')
            
            acc1 = cm1[pe1][pe1]
            acc2 = cm2[pe2][pe2]
            
            # Store results
            if cat1==cat2:
                cat0 = 'same'
            else:
                cat0='diff-' + cat2 
            results_2.append({'subj': subj1, 'cat1': cat1, 'cat2': cat2, 'cat-condition':cat0, \
                            'sim': mssim, 'mse':mse, 'acc1': acc1, 'acc2': acc2})


# Convert results to DataFrame for easier manipulation
sim_diff_2_df = pd.DataFrame(results_2)

### analysis and visualization

In [ ]:
# binary choice task
fig, ax = plt.subplots(figsize=(5*cm,5*cm))
plist = []
# this_df = sim_diff_df[sim_diff_df['matrix']=='cosine']
this_df = sim_diff_1_df
palette=['#DEAC80', cmap_pe['nonPE'], cmap_pe['RPE'], cmap_pe['SPE']]

# this_df = cam_sim_acc_df
this_df.reset_index()
# sns.lineplot(data=this_df, x='cat', y='sim',units='subj', estimator=None,)
violins = sns.violinplot(ax=ax, data=this_df, x='cat1', y='sim',order=['nonPE','RPE','SPE'], \
     hue='cat-condition', \
          hue_order=['same', 'diff-nonPE', 'diff-RPE', 'diff-SPE'],\
     palette=palette,\
     inner="quart", cut=0)

pairs = [(('nonPE','same'), ('nonPE','diff-RPE')),\
          (('nonPE','same'), ('nonPE','diff-SPE')),\
          (('RPE','same'), ('RPE','diff-nonPE')),\
          (('RPE','same'), ('RPE','diff-SPE')),\
          (('SPE','same'), ('SPE','diff-nonPE')),\
          (('SPE','same'), ('SPE','diff-RPE'))]
    
    
# Get the individual violin plots
a = 0
for i, artist in enumerate(violins.collections):
    # Identify the condition and significance for this violin
    # The collections list includes both the violin body and the inner parts (like the bars),
    # so we skip the inner parts by using step 2
    if i % 3 != 0:
        cat = pairs[a][0][0]
        condition = pairs[a][1][1]
        x = this_df[(this_df['cat1']==cat)&(this_df['cat-condition']=='same')]['sim']
        y = this_df[(this_df['cat1']==cat)&(this_df['cat-condition']==condition)]['sim']
        _, p = wilcoxon(x,y, alternative='two-sided')
     #    print(x,y, p)
        a += 1
        plist.append(p)
        if p>0.05: 
            # Set face color to white (or None) to simulate an empty violin
            artist.set_facecolor('none')
            artist.set_edgecolor(palette[condition])  # Outline color only
            artist.set_linewidth(2)  # Optional: thicker outline


ax.set_xticklabels(ax.get_xticklabels(), fontsize=10)
ax.set_xlabel('PE type', fontsize=10, fontweight='bold')
ax.set_ylim(-0.6,1)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
ax.set_ylabel('Similarity', fontsize=10, fontweight='bold')
# Access the existing legend and modify it
# handles, labels = ax.get_legend_handles_labels()  # Get current legend handles and labels
# ax.legend(handles, labels, title='condition', \
#      bbox_to_anchor=(1,0.7),title_fontsize=14, fontsize=14)
ax.get_legend().remove()

plt.title('Binary Choice Task', fontsize=10, fontweight='bold')
plt.savefig('../../results/figS3bLeft', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(plist)
multipletests(plist, method='fdr_bh')

In [ ]:
# navigation task
palette = {
    'same': cmap_pe['same'],
    'diff-nonPE': cmap_pe['nonPE'],
    'diff-SPE': cmap_pe['SPE'],
    'diff-RPE': cmap_pe['RPE'],
    'diff-dualPE': cmap_pe['dualPE']
}

fig, ax = plt.subplots(figsize=(8*cm,5*cm))

# this_df = sim_diff_df[sim_diff_df['matrix']=='cosine']
this_df = sim_diff_2_df
plist = []

# this_df = cam_sim_acc_df
this_df.reset_index()
# sns.lineplot(data=this_df, x='cat', y='sim',units='subj', estimator=None,)
violins = sns.violinplot(ax=ax, data=this_df, x='cat1', y='sim', order=['nonPE','RPE','SPE','dualPE'], \
     hue='cat-condition', \
          hue_order=['same', 'diff-nonPE', 'diff-RPE', 'diff-SPE','diff-dualPE'],\
     palette=[palette['same'], palette['diff-nonPE'], palette['diff-RPE'], palette['diff-SPE'], palette['diff-dualPE']],\
     inner="quart", cut=0)

pairs = [(('nonPE','same'), ('nonPE','diff-RPE')),\
         (('nonPE','same'), ('nonPE','diff-SPE')),\
          (('nonPE','same'), ('nonPE','diff-dualPE')),\
          
          (('RPE','same'), ('RPE','diff-nonPE')),\
          (('RPE','same'), ('RPE','diff-SPE')),\
          (('RPE','same'), ('RPE','diff-dualPE')),\
          
          (('SPE','same'), ('SPE','diff-nonPE')),\
          (('SPE','same'), ('SPE','diff-RPE')),\
          (('SPE','same'), ('SPE','diff-dualPE')),\
          
          (('dualPE','same'), ('dualPE','diff-nonPE')),\
          (('dualPE','same'), ('dualPE','diff-RPE')),\
          (('dualPE','same'), ('dualPE','diff-SPE'))]

    
# Get the individual violin plots
a = 0
for i, artist in enumerate(violins.collections):
    # Identify the condition and significance for this violin
    # The collections list includes both the violin body and the inner parts (like the bars),
    # so we skip the inner parts by using step 2
    if i % 4 != 0:
        cat = pairs[a][0][0]
        condition = pairs[a][1][1]
        x = this_df[(this_df['cat1']==cat)&(this_df['cat-condition']=='same')]['sim']
        y = this_df[(this_df['cat1']==cat)&(this_df['cat-condition']==condition)]['sim']
        _, p = wilcoxon(x,y, alternative='two-sided')
        a += 1
        plist.append(p)
        if p>0.05: 
            # Set face color to white (or None) to simulate an empty violin
            artist.set_facecolor('none')
            artist.set_edgecolor(palette[condition])  # Outline color only
            artist.set_linewidth(2)  # Optional: thicker outline



ax.set_ylim(-0.6,1)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=8)
ax.set_xlabel('PE type', fontsize=8, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
ax.set_ylabel('Similarity', fontsize=10, fontweight='bold')
# # Access the existing legend and modify it
handles, labels = ax.get_legend_handles_labels()  # Get current legend handles and labels
ax.legend(handles, labels, title='condition', \
     bbox_to_anchor=(1,0.7),title_fontsize=10, fontsize=10)
# ax.get_legend().remove()

plt.title('Navigation Task', fontsize=10, fontweight='bold')
plt.savefig('../../results/figS3bRight', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(plist)
multipletests(plist, method='fdr_bh')

## Activation in electrodes

### binary choice task

In [ ]:
ch_pick_1 = [7, 56]
ch_name_1 = ['AFz', 'FC6']
band_pick_1 = [1, 2]

In [ ]:
# topo + cam
pe_list = ['RPE', 'SPE']

info_ = info

fig, ax = plt.subplots(nrows=1, ncols=len(pe_list), figsize=(8*cm,4*cm), sharey=True)



this_df = subj_mean_cam_df

for pe in range(len(pe_list)):
    axin = []
    df = this_df[(this_df['cat']==pe_list[pe])&(this_df['ch']==ch_pick_1[pe])]

    sns.pointplot(ax=ax[pe], data=df, x='bank', y='norm_cam', \
                    color=cmap_pe[pe_list[pe]], capsize=.1,ci=95,\
                          scale=.3,errwidth=.7)
    
    for s in range(len(subj_list)):
        this_cam = df[df['subj']==subj_list[s]]['norm_cam']
        ax[pe].plot(range(len(band_list)), this_cam, \
                    color=cmap_pe[pe_list[pe]], alpha=0.2,linewidth=.5)
        
    line = ax[pe].lines[0]  # This should be the line object corresponding to the points

    # Get the data from the line object
    x_data = line.get_xdata()
    y_data = line.get_ydata()

    # Manually adjust the marker color (since collections may not be directly accessible)
    ax[pe].scatter(x_data, y_data, marker='o', s=5, facecolor='white', \
                   color=cmap_pe[pe_list[pe]]).set_zorder(3) 

    # Print the position of the third point (index 2)
    point_index = band_pick_1[pe]  # Change to the desired point index
    ax[pe].scatter(x_data[point_index], y_data[point_index], \
                   color=cmap_pe[pe_list[pe]], marker='o', s=5).set_zorder(4) 

    


    # ax.set_title('gradCAM (Grand Average)', fontsize=20, fontweight='bold')
    ax[pe].set_xticks(ticks=range(len(band_list_simple)),\
                labels=band_list_simple, fontsize=8)
    ax[pe].set_xlabel('Band', fontsize=8)
    # Format y-axis ticks to 2 decimal places
    ax[pe].set_ylim(0, 1)
    ax[pe].yaxis.set_major_formatter(FuncFormatter(lambda y, _: '{:.1f}'.format(y)))
    
    ax[pe].set_xlim(-.5,4.5)

    if pe == 0:
        ax[pe].set_ylabel('Activation ratio', fontsize=8)
    else:
        ax[pe].set_ylabel('')

    ax[pe].set_title(ch_name_1[pe], fontsize=8, fontweight='bold')
    # Access the existing legend and modify it
    # handles, labels = ax[pe].get_legend_handles_labels()  # Get current legend handles and labels
    # ax[pe].legend(handles, labels, title='PE type', title_fontsize=16, fontsize=16)
    # ax[pe].set_title(f'n_block = {n_block+1}', fontsize=18)

plt.tight_layout()
plt.savefig('../../results/fig5bTop', dpi=300, bbox_inches='tight')
plt.show()

### navigation task

In [ ]:
ch_pick_2 = [7, 56]
tenten_ch_2 = ['AFz','FC6']
band_pick_2 = [1,2]

In [ ]:
this_pe_list_2 = ['RPE', 'SPE']
fig, ax = plt.subplots(nrows=1, ncols=len(this_pe_list_2),figsize=(8*cm,4*cm),sharey=True)


this_df = subj_mean_cam_2_df

for pe in range(len(this_pe_list_2)):
    axin = []
    df = this_df[(this_df['cat']==this_pe_list_2[pe])&(this_df['ch']==ch_pick_2[pe])]

    sns.pointplot(ax=ax[pe], data=df, x='bank', y='norm_cam', \
                    color=cmap_pe[this_pe_list_2[pe]], capsize=.1,\
                          ci=95,\
                          scale=.3,errwidth=.7)

    

    line = ax[pe].lines[0]  # This should be the line object corresponding to the points

    # Get the data from the line object
    x_data = line.get_xdata()
    y_data = line.get_ydata()

    # Manually adjust the marker color (since collections may not be directly accessible)
    ax[pe].scatter(x_data, y_data, marker='o', s=5, facecolor='white', \
                   color=cmap_pe[this_pe_list_2[pe]]).set_zorder(3)

    # Print the position of the third point (index 2)
    point_index = band_pick_2[pe]  # Change to the desired point index
    ax[pe].scatter(x_data[point_index], y_data[point_index], \
                   color=cmap_pe[this_pe_list_2[pe]], marker='o', s=5).set_zorder(4) 

    for s in range(len(subj_list)):
        this_cam = df[df['subj']==subj_list[s]]['norm_cam']
        ax[pe].plot(range(len(band_list)), this_cam, \
                    color=cmap_pe[this_pe_list_2[pe]], alpha=0.2,linewidth=.5)

    
    ax[pe].set_title(tenten_ch_2[pe], fontsize=8, fontweight='bold')
    ax[pe].set_xlim(-.5,4.5)
    ax[pe].set_xticks(ticks=range(len(band_list_simple)),\
                labels=band_list_simple, fontsize=8)
    ax[pe].set_xlabel('Band', fontsize=8)
    

    ax[pe].set_yticks(ticks=ax[pe].get_yticks(),\
                labels=ax[pe].get_yticks(), fontsize=8)
    ax[pe].yaxis.set_major_formatter(FuncFormatter(lambda y, _: '{:.1f}'.format(y)))
    ax[pe].set_ylim(0, 1)
    ax[pe].set_ylabel('Activation map', fontsize=8)
    
    
    if pe == 0:
        ax[pe].set_ylabel('gradCAM', fontsize=8)
    else:
        ax[pe].set_ylabel('')
    # Access the existing legend and modify it
    # handles, labels = ax[pe].get_legend_handles_labels()  # Get current legend handles and labels
    # ax[pe].legend(handles, labels, title='PE type', title_fontsize=16, fontsize=16)
    # ax[pe].set_title(f'n_block = {n_block+1}', fontsize=18)

plt.tight_layout()
plt.savefig('../../results/fig5bBottom', dpi=300, bbox_inches='tight')
plt.show()

## Jaccard similarity

In [ ]:
def jaccard_similarity(x, y):
    return len(set(x).intersection(set(y))) / len(set(x).union(set(y)))

In [ ]:
subj_norm_mean_cam_df = pd.concat([subj_mean_cam_df, subj_mean_cam_2_df])

In [ ]:
all_norm_mean_cam_df = pd.concat([norm_mean_cam_df,norm_mean_cam_2_df])

In [ ]:
# Compute the rank of 'ch' based on 'cam' within the same 'game', 'cat', and 'bank'
df = subj_norm_mean_cam_df
df['rank'] = df.groupby(['subj', 'game', 'cat', 'bank'])['norm_cam'].rank(ascending=False)

# Sort by 'rank' within each group and collect 'ch' into lists
ranked_ch_by_subj_df = (
    df.sort_values(['subj', 'game', 'cat', 'bank', 'rank'])  # Sort by group and rank
    .groupby(['subj','game', 'cat', 'bank'])['ch']          # Group by the columns
    .apply(list)                                     # Collect 'ch' as a list
    .reset_index(name='ranked_ch')                  # Reset index for better readability
)

In [ ]:
# Compute the rank of 'ch' based on 'cam' within the same 'game', 'cat', and 'bank'
df = all_norm_mean_cam_df
df['rank'] = df.groupby(['game', 'cat', 'bank'])['norm_cam'].rank(ascending=False)

# Sort by 'rank' within each group and collect 'ch' into lists
ranked_ch_all_mean_df = (
    df.sort_values(['game', 'cat', 'bank', 'rank'])  # Sort by group and rank
    .groupby(['game', 'cat', 'bank'])['ch']          # Group by the columns
    .apply(list)                                     # Collect 'ch' as a list
    .reset_index(name='ranked_ch')                  # Reset index for better readability
)

In [ ]:
# J(gradCAM): jaccard sim between 1D and 2D Task
grad_ch_rank_sim = {}
sim_set = []

for pe in ['SPE','RPE']:
    pe_ch_rank_sim = []
    for s in subj_list:
        s_ch_rank_sim = []
        for b in range(len(band_list)):
            this_df = ranked_ch_by_subj_df[(ranked_ch_by_subj_df['subj']==s)&\
                                           (ranked_ch_by_subj_df['cat']==pe)&(ranked_ch_by_subj_df['bank']==b)]
            set1 = this_df[this_df['game']=='Game1']['ranked_ch'].iloc[0]
            set2 = this_df[this_df['game']=='Game2']['ranked_ch'].iloc[0]

            b_ch_rank_sim = []
            for n in range(len(set1)):
                js = jaccard_similarity(set1[:n+1], set2[:n+1])
                b_ch_rank_sim.append(js)
                sim_set.append({'subj': s, 'pe': pe, 'band': b, 'num':n, 'ja_sim':js})

            s_ch_rank_sim.append(b_ch_rank_sim)
        pe_ch_rank_sim.append(s_ch_rank_sim)
    grad_ch_rank_sim[pe] = pe_ch_rank_sim

In [ ]:
# J(rnd): jaccard sim for random rank (5 folds)
rnd_rank_sim = []
for s_ in range(len(subj_list)):
    set1_rnd = random.sample(range(64), 64)
    set2_rnd = random.sample(range(64), 64)
    f_rnd_rank_sim = []
    for b_ in range(len(band_list)):
        for n in range(len(set1)):
            js = jaccard_similarity(set1_rnd[:n+1], set2_rnd[:n+1])
            f_rnd_rank_sim.append(js)
            sim_set.append({'subj': s_, 'pe': 'random', 'band': b_, 'num':n, 'ja_sim':js})


    rnd_rank_sim.append(f_rnd_rank_sim)

In [ ]:
js_sim_df = pd.DataFrame(sim_set)

In [ ]:
cm = 1/2.54
fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(15*cm,4.5
                                                *cm), sharey=True)


for b in range(len(band_list)):
    this_df = js_sim_df[js_sim_df['band']==b]
    sns.lineplot(ax=axes[b], data=this_df, x='num', y='ja_sim', hue='pe', hue_order=['RPE','SPE','random'],\
                 palette=[cmap_pe['RPE'], cmap_pe['SPE'], 'black'])
    axes[b].get_legend().remove()
    axes[b].set_title(band_list_simple[b], fontsize=10)
    axes[b].set_xticks([0, 31, 63], [1, 32, 64], fontsize=8)
    axes[b].set_yticks([0, 0.5, 1], [0, 0.5, 1], fontsize=8)
    axes[b].set_ylabel('Jaccard index', fontsize=8)


plt.tight_layout()
plt.savefig('../../results/figS3c.png', dpi=300, bbox_inches='tight')
plt.show()

## Similarity across tasks

#### across tasks: sim_diff_3_df

In [ ]:
# similarity acros categories (between different categories)
df = cam_df
df = df[df['cat']!='dualPE']
subj_list = list(df['subj'].unique())

# Group by 'subj' and 'cat'
grouped = df.groupby(['subj', 'cat', 'game'])

# Extract data for each category
category_data = {}
for (subj, cat, game), group in grouped:
    group['f_e_id'] = group['fold'].astype(str) + '_' + group['e_id'].astype(str)
    group['c_b'] = group['ch'].astype(str) + '_' + group['bank'].astype(str)
    pivot_table = group.pivot(index='f_e_id', columns='c_b', values='cam').fillna(0)
    data_matrix = pivot_table.values
    category_data[(subj, cat, game)] = data_matrix.reshape(data_matrix.shape[0],len(ch_names),len(band_list))

In [ ]:
# Compute similarity between different categories, about 35.4s
results_3 = []

# Compare each category with every other category
for (subj1, cat1, game1), data_matrix1 in category_data.items():
    for (subj2, cat2, game2), data_matrix2 in category_data.items():
        if (subj1 == subj2) & (cat1 == cat2):
            # Compute similarities between different categories
            mssim, mse = compute_ssim_mse(data_matrix1, data_matrix2)
           
            # Get confusion matrix for accuracy
            s1 = subj_list.index(subj1)
            s2 = subj_list.index(subj2)
            this_pe_list = pe_list_1
            pe1 = this_pe_list.index(cat1)
            pe2 = this_pe_list.index(cat2)
            
            cm1 = confusion_matrix(y_true=[item for l in all_true_label_1[s1] for item in l],
                                   y_pred=[item for l in all_pred_label_1[s1] for item in l],
                                   labels=[0, 1, 2], normalize='true')
            
            cm2 = confusion_matrix(y_true=[item for l in all_true_label_1[s2] for item in l],
                                   y_pred=[item for l in all_pred_label_1[s2] for item in l],
                                   labels=[0, 1, 2], normalize='true')
            
            acc1 = cm1[pe1][pe1]
            acc2 = cm2[pe2][pe2]
            
            # Store results
            if game1==game2:
                game0 = 'same'
            else:
                game0 = 'diff'
                
            results_3.append({'subj': subj1, 'cat': cat1, \
                            'game1': game1, 'game2':game2, 'game-condition':game0, \
                            'sim': mssim, 'mse': mse, 'acc1': acc1, 'acc2': acc2})

# Convert results to DataFrame for easier manipulation
sim_diff_3_df = pd.DataFrame(results_3)

In [ ]:
sim_diff_3_mean = sim_diff_3_df.groupby(['subj','cat','game-condition'])['sim'].mean()
sim_diff_3_mean_df = sim_diff_3_mean.reset_index()

### visualization

In [ ]:
# across tasks
cm = 1/2.54 
fig, ax = plt.subplots(figsize=(5*cm, 5.2*cm))

# this_df = sim_diff_df[sim_diff_df['matrix']=='cosine']
df = sim_diff_3_mean_df
cmap_sd = ['#DEAC80','#87694E']

# this_df = cam_sim_acc_df
df.reset_index()


sns.pointplot(ax=ax,\
               data=df, x='cat', y='sim',order=pe_list_1,\
               hue='game-condition', hue_order=['same','diff'], dodge=0.4,
               palette=cmap_sd, join=False,
     capsize=0.1, errwidth=.9, scale=.4, # Error bar caps
     errorbar=('ci',68))
        

# Plot lines connecting the same 'bank' points for each subject
for subj in df['subj'].unique():
     subj_data = df[df['subj'] == subj]
     a = 0
     for cat in pe_list_1:
          cat_data = subj_data[subj_data['cat'] == cat]
          c1_data = cat_data[cat_data['game-condition']=='same']['sim'].mean()
          c2_data = cat_data[cat_data['game-condition']=='diff']['sim'].mean()
     
          ax.plot([a-0.2,a+0.2], \
                    [c1_data,c2_data], color='grey', alpha=.2,\
                         linestyle='--', linewidth=1)
          
          ax.scatter([a-0.2,a+0.2], \
                    [c1_data,c2_data],\
                         edgecolors='None', alpha=.2, color='grey', s=10).set_zorder(0)
          a +=1

               
ax.set_ylim([-.1,1])

ax.set_xticklabels(ax.get_xticklabels(), fontsize=8)
ax.set_xlabel('PE type', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
ax.set_ylabel('Similarity', fontsize=8)
# Access the existing legend and modify it
handles, labels = ax.get_legend_handles_labels()  # Get current legend handles and labels
ax.legend(handles, ['Same', 'Different'], title='Condition', \
     bbox_to_anchor=(1,0.7),title_fontsize=8, fontsize=8)
# ax.get_legend().remove()

# plt.tight_layout()
plt.savefig('../../results/fig5c.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import ttest_1samp
import statsmodels

df = sim_diff_3_mean_df[sim_diff_3_mean_df['game-condition']=='diff']
plist = []
for pe in pe_list_1:
    print(pe)
    this_df = df[(df['cat']==pe)]
    print(cal_mean_sem(this_df['sim']))
    t, p = ttest_1samp(this_df['sim'], 0,alternative='greater')
    print(t, p)
    plist.append(p)

statsmodels.stats.multitest.multipletests(plist, alpha=0.05, method='hs')

## Activation in electrodes across tasks

In [ ]:
sig_list_1 = []

for pe in range(len(pe_list_1)):
    for ch in range(len(ch_names)):
        for bd in range(5):
            sig_list_1.append({'pe_type': pe_list_1[pe], 'ch': ch_names[ch],\
                               'band': bd,\
                                  'CAM_sig': 1 if bd==all_peak_freq_1[pe][ch] else 0})
            
sig_1_df = pd.DataFrame(sig_list_1)
sig_1_df['game'] = 'Binary'

In [ ]:
sig_list_2 = []

for pe in range(len(pe_list_2)):
    for ch in range(len(ch_names)):
        for bd in range(5):
            sig_list_2.append({'pe_type': pe_list_2[pe], 'ch': ch_names[ch], \
                               'band': bd,\
                            'CAM_sig': 1 if bd==all_peak_freq_2[pe][ch] else 0})
            
sig_2_df = pd.DataFrame(sig_list_2)
sig_2_df['game'] = 'Navigation'

In [ ]:
all_sig_df = pd.concat([sig_1_df, sig_2_df])

In [ ]:
# [final] overlaps across games
pe_list = ['RPE','RPE','SPE']
band_choice = [1,3,2]
g_list = ['Binary','Navigation']
info_ = info
colors = cmap_pe

fig, axes = plt.subplots(nrows=len(band_choice), ncols=len(g_list), \
                         figsize=(2*len(g_list)*cm,3*len(band_choice)*cm))


for n in range(len(pe_list)):
    #[left, bottom, width, height]
    mask_params = dict(marker='o',\
                        markersize=3, \
                        markerfacecolor=colors[pe_list[n]], \
                            markeredgecolor=colors[pe_list[n]], alpha=.7)
    
    this_df = all_sig_df[(all_sig_df['pe_type']==pe_list[n])&(all_sig_df['band']==band_choice[n])]

    for g in range(len(g_list)):
        sig_mask = np.array(this_df[this_df['game']==g_list[g]]['CAM_sig'])
        if sig_mask.any():
            im, _ = mne.viz.plot_topomap(64*[0], \
                                    info_,\
                                    mask=sig_mask, mask_params=mask_params,\
                                    sensors=False,
                                    extrapolate='head',\
                                    cmap='Greys',\
                                    # sphere=(0.0, 0.0, 0.0, 0.1),\
                                axes=axes[n,g], show=False)
            print(pe_list[n], g+1, band_choice[n], np.where(sig_mask))
        if n==0:
            axes[n,g].set_title(f'{g_list[g]}', y=1.1,\
                                    fontsize=8, fontweight='bold')
    
    axes[n,0].text(x=0,s=f'{band_list_simple[band_choice[n]]}',
                    y=(axes[n,0].get_position().ymax+axes[n,0].get_position().ymin)/2,\
                horizontalalignment='center',\
    verticalalignment='center', transform=axes[n,0].transAxes,
    fontsize=8)



# plt.tight_layout()
plt.tight_layout(pad=0.1, w_pad=0)
plt.subplots_adjust(left=0.01, right=0.99, top=0.8, bottom=0)
plt.savefig('../../results/fig5d', dpi=300, bbox_inches='tight')
plt.show()